In [2]:
"""
method1_prepare_training_data_combined.py

Prepare Method 1 solve-time training data for the FINAL combined
MINLPLib + QPLIB benchmark.

Inputs
------
1) summary_combined_minlplib_qplib_BB_clean.csv
2) features_nmnnz_combined_clean.csv

Join key
--------
instance_key = SOURCE||instance_name

Do NOT join on instance_index alone: MINLPLib and QPLIB have independent
instance_index spaces.

Method 1 features
-----------------
n, m, nnz

Targets
-------
lp_total_time, sdp_total_time

The script also carries node-count / node-budget columns needed by
method1_cell1_training_combined.py.
"""

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

ROOT = Path(r"C:\Users\suuser\Desktop\minlplib")

FINAL_DIR = (
    ROOT
    / "Areview"
    / "combined_minlplib_qplib"
)

SUMMARY_PATH = (
    FINAL_DIR
    / "summary_combined_minlplib_qplib_BB_clean.csv"
)

FEATURES_PATH = (
    FINAL_DIR
    / "features_nmnnz_combined_clean.csv"
)

OUTPUT_FILE = (
    FINAL_DIR
    / "training_data_method1_combined_minlplib_qplib.csv"
)

JOIN_KEY = "instance_key"
MAX_N_ALLOWED = 300


# ============================================================
# HELPERS
# ============================================================

def _ts():
    return datetime.now().strftime("%H:%M:%S")


def read_any(path):
    p = Path(path)

    if not p.exists():
        raise FileNotFoundError(
            f"File not found: {p}"
        )

    if p.suffix.lower() == ".parquet":
        return pd.read_parquet(p)

    if p.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(p)

    if p.suffix.lower() == ".csv":
        return pd.read_csv(p)

    raise ValueError(
        f"Unsupported file type: {p.suffix}"
    )


def _normalize_key_col(df, name):
    if JOIN_KEY not in df.columns:
        raise ValueError(
            f"{name} has no {JOIN_KEY} column. "
            f"Available columns: {list(df.columns)}"
        )

    out = df.copy()
    out[JOIN_KEY] = (
        out[JOIN_KEY]
        .astype(str)
        .str.strip()
    )

    if out[JOIN_KEY].eq("").any():
        raise ValueError(
            f"{name} contains empty {JOIN_KEY} values."
        )

    if out[JOIN_KEY].duplicated().any():
        dup = (
            out.loc[
                out[JOIN_KEY].duplicated(
                    keep=False
                ),
                JOIN_KEY,
            ]
            .drop_duplicates()
            .head(20)
            .tolist()
        )
        raise ValueError(
            f"{name} has duplicate {JOIN_KEY} values: {dup}"
        )

    return out


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    print("=" * 72, flush=True)
    print(
        f"[{_ts()}] METHOD 1 — PREPARE COMBINED TRAINING DATA",
        flush=True,
    )
    print(f"[{_ts()}] Summary  : {SUMMARY_PATH}", flush=True)
    print(f"[{_ts()}] n,m,nnz  : {FEATURES_PATH}", flush=True)
    print(f"[{_ts()}] Output   : {OUTPUT_FILE}", flush=True)
    print(f"[{_ts()}] Join key : {JOIN_KEY}", flush=True)
    print("=" * 72, flush=True)

    df_summary = _normalize_key_col(
        read_any(SUMMARY_PATH),
        "summary",
    )

    df_features = _normalize_key_col(
        read_any(FEATURES_PATH),
        "n,m,nnz feature file",
    )

    print(
        f"[{_ts()}] Loaded summary  : {len(df_summary)} rows",
        flush=True,
    )
    print(
        f"[{_ts()}] Loaded n,m,nnz  : {len(df_features)} rows",
        flush=True,
    )

    # --------------------------------------------------------
    # HARD SET-EQUALITY CHECK BEFORE MERGE
    # --------------------------------------------------------
    summary_keys = set(
        df_summary[JOIN_KEY].tolist()
    )
    feature_keys = set(
        df_features[JOIN_KEY].tolist()
    )

    missing_features = sorted(
        summary_keys - feature_keys
    )
    extra_features = sorted(
        feature_keys - summary_keys
    )

    if missing_features:
        raise RuntimeError(
            "Summary instances missing from n,m,nnz file. "
            f"First missing: {missing_features[:20]}"
        )

    if extra_features:
        raise RuntimeError(
            "n,m,nnz file contains instances not in final clean summary. "
            f"First extras: {extra_features[:20]}"
        )

    # --------------------------------------------------------
    # USE n,m,nnz FROM THE SOURCE-QCQP STRUCTURAL EXTRACTOR
    # --------------------------------------------------------
    required_feature_cols = [
        JOIN_KEY,
        "n",
        "m",
        "nnz",
    ]

    missing = [
        c for c in required_feature_cols
        if c not in df_features.columns
    ]

    if missing:
        raise ValueError(
            f"n,m,nnz file missing columns: {missing}"
        )

    # The combined summary itself can contain n and a standardized m.
    # For Method 1, use the exact n,m,nnz values produced by the
    # structural extractor, so remove overlapping copies first.
    drop_from_summary = [
        c for c in ["n", "m", "nnz"]
        if c in df_summary.columns
    ]

    df_summary_for_merge = df_summary.drop(
        columns=drop_from_summary
    )

    df_merged = df_summary_for_merge.merge(
        df_features[
            required_feature_cols
        ],
        on=JOIN_KEY,
        how="inner",
        validate="one_to_one",
    )

    if len(df_merged) != len(df_summary):
        raise RuntimeError(
            f"Merge changed row count: "
            f"summary={len(df_summary)}, merged={len(df_merged)}."
        )

    # --------------------------------------------------------
    # NUMERIC / SIZE / TARGET AUDITS
    # --------------------------------------------------------
    numeric_cols = [
        "n",
        "m",
        "nnz",
        "lp_total_time",
        "sdp_total_time",
        "lp_node_count",
        "sdp_node_count",
        "lp_node_budget",
        "sdp_node_budget",
    ]

    missing = [
        c for c in numeric_cols
        if c not in df_merged.columns
    ]

    if missing:
        raise ValueError(
            "Combined training data is missing columns required "
            f"by the next stage: {missing}"
        )

    for col in numeric_cols:
        df_merged[col] = pd.to_numeric(
            df_merged[col],
            errors="coerce",
        )

    if df_merged[numeric_cols].isna().any().any():
        bad_cols = (
            df_merged[numeric_cols]
            .columns[
                df_merged[numeric_cols]
                .isna()
                .any()
            ]
            .tolist()
        )
        raise ValueError(
            f"Missing numeric values after merge in: {bad_cols}"
        )

    if (df_merged["n"] > MAX_N_ALLOWED).any():
        bad = df_merged.loc[
            df_merged["n"] > MAX_N_ALLOWED,
            [
                JOIN_KEY,
                "n",
            ],
        ]
        raise RuntimeError(
            f"Unfiltered n > {MAX_N_ALLOWED} instances entered "
            "Method 1 training data:\n"
            + bad.to_string(index=False)
        )

    if (df_merged[["n", "m", "nnz"]] < 0).any().any():
        raise ValueError(
            "Negative structural feature value found."
        )

    if (
        df_merged[
            ["lp_total_time", "sdp_total_time"]
        ]
        <= 0
    ).any().any():
        raise ValueError(
            "Non-positive LP/SDP total solve time found."
        )

    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------
    OUTPUT_FILE.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    df_merged = (
        df_merged
        .sort_values(
            ["source", "instance_name"]
            if {
                "source",
                "instance_name",
            }.issubset(df_merged.columns)
            else [JOIN_KEY]
        )
        .reset_index(drop=True)
    )

    df_merged.to_csv(
        OUTPUT_FILE,
        index=False,
    )

    print("\n" + "=" * 72, flush=True)
    print(
        f"[{_ts()}] Saved -> {OUTPUT_FILE}",
        flush=True,
    )
    print(
        f"[{_ts()}] Shape : {df_merged.shape}",
        flush=True,
    )

    if "source" in df_merged.columns:
        print(
            f"\n[{_ts()}] Rows by source:",
            flush=True,
        )
        print(
            df_merged[
                "source"
            ].value_counts().to_string(),
            flush=True,
        )

    print(
        f"\n[{_ts()}] Structural features:",
        flush=True,
    )
    print(
        df_merged[
            ["n", "m", "nnz"]
        ].describe().to_string(),
        flush=True,
    )

    print(
        f"\n[{_ts()}] Target times:",
        flush=True,
    )
    print(
        df_merged[
            ["lp_total_time", "sdp_total_time"]
        ].describe().to_string(),
        flush=True,
    )

    print(
        f"\n[{_ts()}] Next step:",
        flush=True,
    )
    print(
        f"[{_ts()}]   Run method1_cell1_training_combined.py",
        flush=True,
    )
    print("=" * 72, flush=True)


[01:33:30] METHOD 1 — PREPARE COMBINED TRAINING DATA
[01:33:30] Summary  : C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\summary_combined_minlplib_qplib_BB_clean.csv
[01:33:30] n,m,nnz  : C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\features_nmnnz_combined_clean.csv
[01:33:30] Output   : C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\training_data_method1_combined_minlplib_qplib.csv
[01:33:30] Join key : instance_key
[01:33:30] Loaded summary  : 283 rows
[01:33:30] Loaded n,m,nnz  : 283 rows

[01:33:30] Saved -> C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\training_data_method1_combined_minlplib_qplib.csv
[01:33:30] Shape : (283, 44)

[01:33:30] Rows by source:
source
MINLPLIB    147
QPLIB       136

[01:33:30] Structural features:
                n            m            nnz
count  283.000000   283.000000     283.000000
mean    97.837456   313.282686    5343.696113
std     81.564414   946.520855   11614.4888

In [4]:
"""
method1_cell1_training_combined.py

METHOD 1 solve-time estimation for the FINAL combined
MINLPLib + QPLIB benchmark.

Key changes from the old MINLPLib-only version
------------------------------------------------
- KEY_COL = instance_key, not instance_index.
- Absolute combined-benchmark paths are used.
- The final clean benchmark is audited again for n <= 300.
- Source/name metadata are preserved in prediction/budget files.
- Time-regression FEATURES remain exactly: n, m, nnz.
- Time-regression TARGETS remain exactly:
      lp_total_time, sdp_total_time

Feature-extraction time is NOT added to these targets. It is a common
preprocessing cost and must be added later in the end-to-end ML runtime
comparison, not used to distort the LP-vs-SDP' node-budget ratio.
"""

from datetime import datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.api.types import CategoricalDtype
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False


# ============================================================
# CONFIG
# ============================================================

SETUP = "DI"
DIM_TAG = "combined_minlplib_qplib"

ROOT = Path(r"C:\Users\suuser\Desktop\minlplib")

FINAL_DIR = (
    ROOT
    / "Areview"
    / "combined_minlplib_qplib"
)

TRAINING_FILE = (
    FINAL_DIR
    / "training_data_method1_combined_minlplib_qplib.csv"
)

KEY_COL = "instance_key"

FEATURE_COLS = [
    "n",
    "m",
    "nnz",
]

TARGETS = [
    "lp_total_time",
    "sdp_total_time",
]

COUNT_BUDGET_COLS = [
    "lp_node_count",
    "sdp_node_count",
    "lp_node_budget",
    "sdp_node_budget",
]

TEST_SIZE = 0.2
RANDOM_STATE = 42
N_FOLDS = 5
OOF_MODEL = "xgb"

REFERENCE_LP_NODES = 100
SDP_NODES_MIN = 1
SDP_NODES_MAX = 100

MAX_N_ALLOWED = 300

OUTPUT_ROOT = (
    FINAL_DIR
    / "reg_outputs_method1"
    / DIM_TAG
)
OUTPUT_ROOT.mkdir(
    exist_ok=True,
    parents=True,
)

RUN_DIR = (
    OUTPUT_ROOT
    / f"{SETUP}__method1_time_regression"
)
RUN_DIR.mkdir(
    exist_ok=True,
    parents=True,
)

BUDGET_FILE = (
    RUN_DIR
    / f"method1_budget_predictions_{DIM_TAG}.csv"
)

CALIBRATION_FILE = (
    RUN_DIR
    / f"full_budget_calibration_subset_{DIM_TAG}.csv"
)


# ============================================================
# HELPERS
# ============================================================

def _ts():
    return datetime.now().strftime("%H:%M:%S")


def read_any(path):
    p = Path(path)

    if not p.exists():
        raise FileNotFoundError(
            f"File not found: {p}"
        )

    if p.suffix.lower() == ".parquet":
        return pd.read_parquet(p)

    if p.suffix.lower() in {
        ".xlsx",
        ".xls",
    }:
        return pd.read_excel(p)

    if p.suffix.lower() == ".csv":
        return pd.read_csv(p)

    raise ValueError(
        f"Unsupported: {p.suffix}"
    )


def make_deterministic_split(
    keys,
    test_frac=0.2,
):
    h = pd.util.hash_pandas_object(
        keys.astype(str),
        index=False,
    ).to_numpy(
        dtype=np.uint64
    )

    buckets = (
        h % 100
    ).astype(
        np.int64
    )

    cutoff = int(
        (1.0 - test_frac)
        * 100
    )

    train_idx = np.where(
        buckets < cutoff
    )[0]

    test_idx = np.where(
        buckets >= cutoff
    )[0]

    if (
        len(train_idx) == 0
        or len(test_idx) == 0
    ):
        raise ValueError(
            "Deterministic holdout split "
            "produced an empty set."
        )

    return (
        train_idx,
        test_idx,
    )


def build_preprocessor(X):
    cat_cols = [
        c
        for c in X.columns
        if (
            pd.api.types.is_object_dtype(
                X[c]
            )
            or isinstance(
                X[c].dtype,
                CategoricalDtype,
            )
        )
    ]

    num_cols = [
        c
        for c in X.columns
        if c not in cat_cols
    ]

    transformers = []

    if cat_cols:
        try:
            ohe = OneHotEncoder(
                sparse_output=False,
                handle_unknown="ignore",
            )
        except TypeError:
            ohe = OneHotEncoder(
                sparse=False,
                handle_unknown="ignore",
            )

        transformers.append(
            (
                "cat",
                Pipeline(
                    [
                        (
                            "imp",
                            SimpleImputer(
                                strategy="most_frequent"
                            ),
                        ),
                        (
                            "ohe",
                            ohe,
                        ),
                    ]
                ),
                cat_cols,
            )
        )

    if num_cols:
        transformers.append(
            (
                "num",
                Pipeline(
                    [
                        (
                            "imp",
                            SimpleImputer(
                                strategy="median"
                            ),
                        ),
                    ]
                ),
                num_cols,
            )
        )

    if not transformers:
        raise ValueError(
            "No usable features found."
        )

    return ColumnTransformer(
        transformers,
        remainder="drop",
    )


def safe_pearson_r(
    y_true,
    y_pred,
):
    yt = np.asarray(
        y_true,
        dtype=float,
    )
    yp = np.asarray(
        y_pred,
        dtype=float,
    )

    mask = (
        np.isfinite(yt)
        & np.isfinite(yp)
    )

    yt = yt[mask]
    yp = yp[mask]

    if (
        len(yt) < 2
        or np.std(yt) <= 0
        or np.std(yp) <= 0
    ):
        return float("nan")

    r = np.corrcoef(
        yt,
        yp,
    )[0, 1]

    return (
        float(r)
        if np.isfinite(r)
        else float("nan")
    )


def mean_absolute_percentage_error(
    y_true,
    y_pred,
):
    yt = np.asarray(
        y_true,
        dtype=float,
    )
    yp = np.asarray(
        y_pred,
        dtype=float,
    )

    mask = (
        np.isfinite(yt)
        & np.isfinite(yp)
        & (
            np.abs(yt)
            > 1e-10
        )
    )

    if mask.sum() == 0:
        return float("nan")

    return float(
        np.mean(
            np.abs(
                (
                    yt[mask]
                    - yp[mask]
                )
                / yt[mask]
            )
        )
        * 100.0
    )


def compute_metrics(
    y_true,
    y_pred,
):
    mse = float(
        mean_squared_error(
            y_true,
            y_pred,
        )
    )

    return {
        "mse": mse,
        "rmse": float(
            np.sqrt(mse)
        ),
        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),
        "mape": mean_absolute_percentage_error(
            y_true,
            y_pred,
        ),
        "pearson_r": safe_pearson_r(
            y_true,
            y_pred,
        ),
    }


def save_scatter(
    y_true,
    y_pred,
    output_path,
    title,
    x_label,
    y_label,
):
    fig, ax = plt.subplots(
        figsize=(5.5, 5.5)
    )

    ax.scatter(
        y_true,
        y_pred,
        s=10,
        alpha=0.4,
    )

    lo = min(
        float(np.min(y_true)),
        float(np.min(y_pred)),
    )

    hi = max(
        float(np.max(y_true)),
        float(np.max(y_pred)),
    )

    ax.plot(
        [lo, hi],
        [lo, hi],
        "r--",
        label="Perfect",
    )

    ax.set_xlabel(
        x_label
    )
    ax.set_ylabel(
        y_label
    )
    ax.set_title(
        title
    )
    ax.legend()

    plt.tight_layout()
    plt.savefig(
        output_path,
        dpi=200,
    )
    plt.close()


def make_model(
    model_name,
    preprocessor,
):
    if model_name == "xgb":
        if not XGBOOST_AVAILABLE:
            raise RuntimeError(
                "XGBoost not installed."
            )

        reg = XGBRegressor(
            n_estimators=400,
            learning_rate=0.08,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    elif model_name == "rf":
        reg = RandomForestRegressor(
            n_estimators=400,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )

    elif model_name == "gbr":
        reg = GradientBoostingRegressor(
            n_estimators=400,
            learning_rate=0.08,
            max_depth=5,
            random_state=RANDOM_STATE,
        )

    else:
        raise ValueError(
            f"Unknown model: {model_name}"
        )

    return Pipeline(
        [
            (
                "pre",
                preprocessor,
            ),
            (
                "model",
                reg,
            ),
        ]
    )


def make_all_eval_models(
    preprocessor,
):
    models = {
        "gbr": make_model(
            "gbr",
            preprocessor,
        ),
        "rf": make_model(
            "rf",
            preprocessor,
        ),
    }

    if XGBOOST_AVAILABLE:
        models["xgb"] = make_model(
            "xgb",
            preprocessor,
        )

    return models


# ============================================================
# INPUT PREPARATION
# ============================================================

def check_required_columns(df):
    required = (
        [KEY_COL]
        + FEATURE_COLS
        + TARGETS
        + COUNT_BUDGET_COLS
    )

    missing = [
        c
        for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            "Training file missing columns: "
            f"{missing}\nAvailable: {list(df.columns)}"
        )


def prepare_input_dataframe(df):
    df = df.copy()

    check_required_columns(
        df
    )

    # Combined benchmark identity.
    df[KEY_COL] = (
        df[KEY_COL]
        .astype(str)
        .str.strip()
    )

    if df[KEY_COL].eq("").any():
        raise ValueError(
            "Some instance_key values are empty."
        )

    if df[KEY_COL].duplicated().any():
        dup = (
            df.loc[
                df[KEY_COL].duplicated(
                    keep=False
                ),
                KEY_COL,
            ]
            .drop_duplicates()
            .head(10)
            .tolist()
        )

        raise ValueError(
            f"Duplicate instance_key values: {dup}"
        )

    for col in (
        FEATURE_COLS
        + TARGETS
        + COUNT_BUDGET_COLS
    ):
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce",
        )

    if df[
        FEATURE_COLS
    ].isna().any().any():
        raise ValueError(
            "Missing feature values."
        )

    if df[
        TARGETS
    ].isna().any().any():
        raise ValueError(
            "Missing target times."
        )

    if df[
        COUNT_BUDGET_COLS
    ].isna().any().any():
        raise ValueError(
            "Missing node-count/budget values."
        )

    if (
        df[
            TARGETS
        ]
        <= 0
    ).any().any():
        raise ValueError(
            "Non-positive target times."
        )

    if (
        df[
            COUNT_BUDGET_COLS
        ]
        <= 0
    ).any().any():
        raise ValueError(
            "Non-positive node-count/budget values."
        )

    # Extra protection against accidental old unfiltered MINLPLib files.
    if (
        df["n"]
        > MAX_N_ALLOWED
    ).any():
        bad = df.loc[
            df["n"]
            > MAX_N_ALLOWED,
            [
                KEY_COL,
                "n",
            ],
        ]

        raise RuntimeError(
            f"n > {MAX_N_ALLOWED} entered combined time training:\n"
            + bad.to_string(
                index=False
            )
        )

    df[
        "is_full_budget_calibration"
    ] = (
        (
            df[
                "lp_node_count"
            ]
            == df[
                "lp_node_budget"
            ]
        )
        & (
            df[
                "sdp_node_count"
            ]
            == df[
                "sdp_node_budget"
            ]
        )
    )

    n_cal = int(
        df[
            "is_full_budget_calibration"
        ].sum()
    )

    if n_cal < N_FOLDS:
        raise ValueError(
            f"Not enough full-budget rows for "
            f"{N_FOLDS}-fold cross-fitting: {n_cal}."
        )

    unique_lp_budgets = sorted(
        df.loc[
            df[
                "is_full_budget_calibration"
            ],
            "lp_node_budget",
        ]
        .astype(int)
        .unique()
        .tolist()
    )

    if (
        len(unique_lp_budgets) == 1
        and unique_lp_budgets[0]
        != REFERENCE_LP_NODES
    ):
        raise ValueError(
            "REFERENCE_LP_NODES mismatch: "
            f"config={REFERENCE_LP_NODES}, "
            f"data={unique_lp_budgets[0]}."
        )

    if len(
        unique_lp_budgets
    ) > 1:
        print(
            f"[{_ts()}] WARNING: "
            f"multiple LP budgets: {unique_lp_budgets}",
            flush=True,
        )

    return (
        df.sort_values(
            KEY_COL
        )
        .reset_index(drop=True)
    )


def metadata_frame(
    df,
):
    cols = [
        c
        for c in [
            KEY_COL,
            "source",
            "instance_name",
            "instance_index",
        ]
        if c in df.columns
    ]

    return df[
        cols
    ].copy()


# ============================================================
# EVALUATION
# ============================================================

def evaluate_models_on_calibration_subset(
    df_all,
    target_name,
    run_dir,
):
    print(
        f"\n{'=' * 58}",
        flush=True,
    )
    print(
        f"[{_ts()}] TARGET: {target_name}",
        flush=True,
    )

    df_cal = (
        df_all[
            df_all[
                "is_full_budget_calibration"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    ids = (
        df_cal[
            KEY_COL
        ]
        .astype(str)
    )

    X = (
        df_cal[
            FEATURE_COLS
        ]
        .copy()
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
    )

    y = (
        df_cal[
            target_name
        ]
        .astype(float)
        .values
    )

    target_dir = (
        run_dir
        / target_name
    )

    target_dir.mkdir(
        exist_ok=True,
        parents=True,
    )

    tr_idx, te_idx = (
        make_deterministic_split(
            ids,
            test_frac=TEST_SIZE,
        )
    )

    X_tr = (
        X.iloc[
            tr_idx
        ]
        .reset_index(drop=True)
    )

    X_te = (
        X.iloc[
            te_idx
        ]
        .reset_index(drop=True)
    )

    y_tr = y[
        tr_idx
    ]

    y_te = y[
        te_idx
    ]

    meta_te = (
        metadata_frame(
            df_cal
        )
        .iloc[
            te_idx
        ]
        .reset_index(drop=True)
    )

    preprocessor = (
        build_preprocessor(
            X_tr
        )
    )

    models = (
        make_all_eval_models(
            preprocessor
        )
    )

    results = {}

    for name, pipe in models.items():
        pipe.fit(
            X_tr,
            y_tr,
        )

        y_hat = pipe.predict(
            X_te
        )

        metrics = compute_metrics(
            y_te,
            y_hat,
        )

        results[name] = {
            "model": name,
            "target": target_name,
            "dim_tag": DIM_TAG,
            "evaluation_subset":
                "full_budget_calibration_only",
            "n_train": len(y_tr),
            "n_test": len(y_te),
            **metrics,
        }

        print(
            f"[{_ts()}] [{name}] "
            f"RMSE={metrics['rmse']:.6f}s "
            f"MAPE={metrics['mape']:.2f}% "
            f"Pearson_r={metrics['pearson_r']:.4f}",
            flush=True,
        )

        pred_df = (
            meta_te.copy()
        )

        pred_df[
            "y_true"
        ] = y_te

        pred_df[
            "y_pred"
        ] = y_hat

        pred_df[
            "abs_err"
        ] = np.abs(
            y_hat
            - y_te
        )

        pred_df.to_csv(
            target_dir
            / (
                f"preds_{target_name}_{name}"
                "_holdout_calibration.csv"
            ),
            index=False,
        )

        save_scatter(
            y_te,
            y_hat,
            target_dir
            / (
                f"scatter_{target_name}_{name}"
                "_holdout_calibration.png"
            ),
            (
                f"Method 1 | {DIM_TAG} | "
                f"{target_name} | {name}"
            ),
            f"True {target_name} (s)",
            f"Predicted {target_name} (s)",
        )

    (
        pd.DataFrame.from_dict(
            results,
            orient="index",
        )
        .reset_index(drop=True)
        .to_csv(
            target_dir
            / (
                f"results_{target_name}"
                "_holdout_calibration.csv"
            ),
            index=False,
        )
    )

    return results


# ============================================================
# CROSS-FIT
# ============================================================

def generate_crossfit_predictions_for_all(
    df_all,
    target_name,
    run_dir,
):
    print(
        f"\n[{_ts()}] "
        f"Cross-fitting ALL clean rows: {target_name}",
        flush=True,
    )

    X_all = (
        df_all[
            FEATURE_COLS
        ]
        .copy()
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
    )

    y_observed = (
        df_all[
            target_name
        ]
        .astype(float)
        .values
    )

    calibration_mask = (
        df_all[
            "is_full_budget_calibration"
        ]
        .astype(bool)
        .to_numpy()
    )

    n_rows = len(
        df_all
    )

    y_pred = np.full(
        n_rows,
        np.nan,
        dtype=float,
    )

    fold_assignment = np.full(
        n_rows,
        -1,
        dtype=int,
    )

    kf = KFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    for fold_no, (
        _,
        pred_positions,
    ) in enumerate(
        kf.split(
            np.arange(
                n_rows
            )
        ),
        start=1,
    ):
        train_mask = (
            calibration_mask.copy()
        )

        train_mask[
            pred_positions
        ] = False

        train_positions = np.where(
            train_mask
        )[0]

        if len(
            train_positions
        ) == 0:
            raise ValueError(
                f"Fold {fold_no}: "
                "no calibration rows."
            )

        X_train = (
            X_all.iloc[
                train_positions
            ]
            .reset_index(drop=True)
        )

        y_train = y_observed[
            train_positions
        ]

        X_predict = (
            X_all.iloc[
                pred_positions
            ]
            .reset_index(drop=True)
        )

        model = make_model(
            OOF_MODEL,
            build_preprocessor(
                X_train
            ),
        )

        model.fit(
            X_train,
            y_train,
        )

        y_pred[
            pred_positions
        ] = model.predict(
            X_predict
        )

        fold_assignment[
            pred_positions
        ] = fold_no

        print(
            f"[{_ts()}]   "
            f"fold={fold_no}/{N_FOLDS} "
            f"fit={len(train_positions):,} "
            f"predict={len(pred_positions):,}",
            flush=True,
        )

    if (
        np.isnan(
            y_pred
        ).any()
        or (
            fold_assignment
            < 1
        ).any()
    ):
        raise RuntimeError(
            "Cross-fit failed: missing predictions."
        )

    target_dir = (
        run_dir
        / target_name
    )

    target_dir.mkdir(
        exist_ok=True,
        parents=True,
    )

    output_file = (
        target_dir
        / f"preds_{target_name}_oof.csv"
    )

    df_pred = metadata_frame(
        df_all
    )

    df_pred[
        "y_true"
    ] = y_observed

    df_pred[
        "y_pred_oof"
    ] = y_pred

    df_pred[
        "abs_err"
    ] = np.abs(
        y_pred
        - y_observed
    )

    df_pred[
        "is_full_budget_calibration"
    ] = calibration_mask

    df_pred[
        "crossfit_fold"
    ] = fold_assignment

    df_pred.to_csv(
        output_file,
        index=False,
    )

    primary_metrics = (
        compute_metrics(
            y_observed[
                calibration_mask
            ],
            y_pred[
                calibration_mask
            ],
        )
    )

    pd.DataFrame(
        [
            {
                "target": target_name,
                "model": OOF_MODEL,
                "dim_tag": DIM_TAG,
                "n_all_clean_rows_predicted":
                    len(df_pred),
                "n_calibration_rows_evaluated":
                    int(
                        calibration_mask.sum()
                    ),
                **primary_metrics,
            }
        ]
    ).to_csv(
        target_dir
        / f"results_{target_name}_oof.csv",
        index=False,
    )

    save_scatter(
        y_observed[
            calibration_mask
        ],
        y_pred[
            calibration_mask
        ],
        target_dir
        / (
            f"scatter_{target_name}"
            "_oof_calibration.png"
        ),
        (
            f"Method 1 cross-fit | "
            f"{DIM_TAG} | {target_name}"
        ),
        f"True {target_name} (s)",
        f"Cross-fit predicted {target_name} (s)",
    )

    print(
        f"[{_ts()}] "
        f"OOF Pearson_r="
        f"{primary_metrics['pearson_r']:.4f} "
        f"RMSE={primary_metrics['rmse']:.6f}s",
        flush=True,
    )

    print(
        f"[{_ts()}] Saved -> {output_file}",
        flush=True,
    )

    return output_file


def train_final_model_on_calibration_subset(
    df_all,
    target_name,
    run_dir,
):
    df_cal = (
        df_all[
            df_all[
                "is_full_budget_calibration"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    X = (
        df_cal[
            FEATURE_COLS
        ]
        .copy()
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
    )

    y = (
        df_cal[
            target_name
        ]
        .astype(float)
        .values
    )

    model = make_model(
        OOF_MODEL,
        build_preprocessor(
            X
        ),
    )

    model.fit(
        X,
        y,
    )

    target_dir = (
        run_dir
        / target_name
    )

    target_dir.mkdir(
        exist_ok=True,
        parents=True,
    )

    model_path = (
        target_dir
        / (
            f"model_{target_name}_"
            f"{OOF_MODEL}_final.pkl"
        )
    )

    joblib.dump(
        model,
        model_path,
    )

    joblib.dump(
        FEATURE_COLS,
        target_dir
        / (
            f"feature_cols_{target_name}.pkl"
        ),
    )

    print(
        f"[{_ts()}] "
        f"Final model saved -> {model_path}",
        flush=True,
    )


# ============================================================
# BUDGET FILE
# ============================================================

def compute_sdp_nodes(
    lp_time_pred,
    sdp_time_pred,
):
    lp = np.clip(
        np.asarray(
            lp_time_pred,
            dtype=float,
        ),
        1e-6,
        None,
    )

    sdp = np.clip(
        np.asarray(
            sdp_time_pred,
            dtype=float,
        ),
        1e-6,
        None,
    )

    return (
        np.clip(
            np.floor(
                lp
                / sdp
                * REFERENCE_LP_NODES
            ),
            SDP_NODES_MIN,
            SDP_NODES_MAX,
        )
        .astype(int)
    )


def build_budget_file(
    lp_oof_path,
    sdp_oof_path,
    output_file,
):
    print(
        f"\n[{_ts()}] "
        "Building Method 1 budget file...",
        flush=True,
    )

    df_lp = pd.read_csv(
        lp_oof_path
    )

    df_sdp = pd.read_csv(
        sdp_oof_path
    )

    required = {
        KEY_COL,
        "y_true",
        "y_pred_oof",
        "is_full_budget_calibration",
        "crossfit_fold",
    }

    if not required.issubset(
        df_lp.columns
    ):
        raise ValueError(
            f"LP file missing: "
            f"{required - set(df_lp.columns)}"
        )

    if not required.issubset(
        df_sdp.columns
    ):
        raise ValueError(
            f"SDP file missing: "
            f"{required - set(df_sdp.columns)}"
        )

    meta_cols = [
        c
        for c in [
            KEY_COL,
            "source",
            "instance_name",
            "instance_index",
        ]
        if c in df_lp.columns
    ]

    lp_cols = (
        meta_cols
        + [
            "y_true",
            "y_pred_oof",
            "is_full_budget_calibration",
            "crossfit_fold",
        ]
    )

    df_lp = (
        df_lp[
            lp_cols
        ]
        .rename(
            columns={
                "y_true":
                    "lp_time_true_observed",
                "y_pred_oof":
                    "lp_time_pred",
                "crossfit_fold":
                    "lp_crossfit_fold",
            }
        )
    )

    sdp_keep = [
        KEY_COL,
        "y_true",
        "y_pred_oof",
        "is_full_budget_calibration",
        "crossfit_fold",
    ]

    df_sdp = (
        df_sdp[
            sdp_keep
        ]
        .rename(
            columns={
                "y_true":
                    "sdp_time_true_observed",
                "y_pred_oof":
                    "sdp_time_pred",
                "is_full_budget_calibration":
                    "is_full_budget_calibration_sdp",
                "crossfit_fold":
                    "sdp_crossfit_fold",
            }
        )
    )

    df_budget = pd.merge(
        df_lp,
        df_sdp,
        on=KEY_COL,
        how="inner",
        validate="one_to_one",
    )

    if not (
        df_budget[
            "is_full_budget_calibration"
        ]
        == df_budget[
            "is_full_budget_calibration_sdp"
        ]
    ).all():
        raise RuntimeError(
            "LP and SDP files disagree "
            "on calibration membership."
        )

    df_budget = (
        df_budget.drop(
            columns=[
                "is_full_budget_calibration_sdp"
            ]
        )
    )

    for col in [
        "lp_time_pred",
        "sdp_time_pred",
    ]:
        df_budget[col] = pd.to_numeric(
            df_budget[col],
            errors="coerce",
        )

    if df_budget[
        [
            "lp_time_pred",
            "sdp_time_pred",
        ]
    ].isna().any().any():
        raise ValueError(
            "Missing Method 1 predictions."
        )

    df_budget[
        "lp_nodes"
    ] = REFERENCE_LP_NODES

    df_budget[
        "sdp_nodes"
    ] = compute_sdp_nodes(
        df_budget[
            "lp_time_pred"
        ].values,
        df_budget[
            "sdp_time_pred"
        ].values,
    )

    df_budget[
        "predicted_time_ratio_lp_over_sdp"
    ] = (
        df_budget[
            "lp_time_pred"
        ].clip(
            lower=1e-6
        )
        / df_budget[
            "sdp_time_pred"
        ].clip(
            lower=1e-6
        )
    )

    df_budget[
        "predicted_time_ratio_sdp_over_lp"
    ] = (
        df_budget[
            "sdp_time_pred"
        ].clip(
            lower=1e-6
        )
        / df_budget[
            "lp_time_pred"
        ].clip(
            lower=1e-6
        )
    )

    df_budget = (
        df_budget
        .sort_values(
            KEY_COL
        )
        .reset_index(drop=True)
    )

    df_budget.to_csv(
        output_file,
        index=False,
    )

    print(
        f"[{_ts()}] Saved -> "
        f"{output_file} "
        f"({len(df_budget):,} rows)",
        flush=True,
    )

    print(
        df_budget[
            "sdp_nodes"
        ].describe().to_string(),
        flush=True,
    )

    return df_budget


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    print("=" * 72, flush=True)
    print(
        f"[{_ts()}] METHOD 1 — "
        "FULL-BUDGET CALIBRATION + "
        "ALL-INSTANCE BUDGETS",
        flush=True,
    )
    print(
        f"[{_ts()}] DIM_TAG={DIM_TAG} "
        f"KEY_COL={KEY_COL} "
        f"OOF_MODEL={OOF_MODEL}",
        flush=True,
    )
    print(
        f"[{_ts()}] Training file: "
        f"{TRAINING_FILE}",
        flush=True,
    )
    print("=" * 72, flush=True)

    df_all = prepare_input_dataframe(
        read_any(
            TRAINING_FILE
        )
    )

    n_all = len(
        df_all
    )

    n_cal = int(
        df_all[
            "is_full_budget_calibration"
        ].sum()
    )

    print(
        f"[{_ts()}] "
        f"all clean={n_all:,} "
        f"calibration={n_cal:,} "
        f"predicted-only={n_all - n_cal:,}",
        flush=True,
    )

    cal_cols = [
        c
        for c in [
            KEY_COL,
            "source",
            "instance_name",
            "instance_index",
            "lp_node_count",
            "lp_node_budget",
            "sdp_node_count",
            "sdp_node_budget",
            "is_full_budget_calibration",
        ]
        if c in df_all.columns
    ]

    df_all[
        cal_cols
    ].to_csv(
        CALIBRATION_FILE,
        index=False,
    )

    all_results = {}
    oof_files = {}

    for target in TARGETS:
        all_results[
            target
        ] = evaluate_models_on_calibration_subset(
            df_all,
            target,
            RUN_DIR,
        )

        oof_files[
            target
        ] = generate_crossfit_predictions_for_all(
            df_all,
            target,
            RUN_DIR,
        )

        train_final_model_on_calibration_subset(
            df_all,
            target,
            RUN_DIR,
        )

    df_budget = build_budget_file(
        lp_oof_path=oof_files[
            "lp_total_time"
        ],
        sdp_oof_path=oof_files[
            "sdp_total_time"
        ],
        output_file=BUDGET_FILE,
    )

    print(
        f"\n[{_ts()}] Budget file -> "
        f"{BUDGET_FILE} "
        f"rows={len(df_budget):,}",
        flush=True,
    )


[01:39:09] METHOD 1 — FULL-BUDGET CALIBRATION + ALL-INSTANCE BUDGETS
[01:39:09] DIM_TAG=combined_minlplib_qplib KEY_COL=instance_key OOF_MODEL=xgb
[01:39:09] Training file: C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\training_data_method1_combined_minlplib_qplib.csv
[01:39:09] all clean=283 calibration=191 predicted-only=92

[01:39:09] TARGET: lp_total_time
[01:39:10] [gbr] RMSE=12.558027s MAPE=48.40% Pearson_r=0.8748
[01:39:10] [rf] RMSE=6.569518s MAPE=39.42% Pearson_r=0.9053
[01:39:11] [xgb] RMSE=7.257402s MAPE=106.33% Pearson_r=0.9044

[01:39:11] Cross-fitting ALL clean rows: lp_total_time
[01:39:11]   fold=1/5 fit=150 predict=57
[01:39:11]   fold=2/5 fit=156 predict=57
[01:39:12]   fold=3/5 fit=146 predict=57
[01:39:12]   fold=4/5 fit=153 predict=56
[01:39:12]   fold=5/5 fit=159 predict=56
[01:39:12] OOF Pearson_r=0.4849 RMSE=16.242419s
[01:39:12] Saved -> C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\reg_outputs_method1\combined_minlplib_qpl

In [1]:
"""Synthetic′ Method 2: preserve original dynamic probes and (y, delta) target formula.

Inputs: postrun-clean Synthetic′ LIVE, 64-feature table, Method-1 predicted
budgets. Joins use instance_key=SYNTHETIC||<original numeric instance ID>.
Outputs remain targets and probe_features for windows 10,20,30.
No new B&B solve is performed.
"""

from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

DIM_TAG = "synthetic_prime"

ROOT = Path(r"C:\Users\suuser\Desktop\minlplib")
BASE_DIR = ROOT / "paperexperiments" / "synthetic_prime_minimal_revision"

FINAL_DIR = BASE_DIR / 'bb_separate_trees' / 'synthetic_prime_8016_fixed100_v1' / 'postrun_clean'

LIVE_PATH = (
    FINAL_DIR
    / "synthetic_prime_BB_LIVE_clean.parquet"
)

FEATURES_PATH = (
    FINAL_DIR
    / "synthetic_prime_DI_64_clean.xlsx"
)

SETUP = "DI"

METHOD1_BUDGET_PATH = (
    FINAL_DIR
    / "reg_outputs_method1"
    / DIM_TAG
    / f"{SETUP}__method1_time_regression"
    / f"method1_budget_predictions_{DIM_TAG}.csv"
)

# Generate every requested early-window dataset in one execution.
# Add/remove values freely, e.g. [10, 20] or [10, 20, 30].
PROBE_WINDOWS = [10, 20, 30]

TAU_BINARY = 0.01
ETA = 1e-3
EPS = 1e-9
EPS_CV = 1e-6

MAX_N_ALLOWED = None  # No Synthetic′ size cutoff

# Preserve the original dynamic-feature definitions.
STAT_NAMES = [
    "min",
    "max",
    "avg",
    "mu3",
    "IQR",
    "Pout",
    "CV",
]

FRONTIER_STAT_NAMES = [
    "min",
    "avg",
    "mu3",
    "IQR",
    "Pout",
    "CV",
]

PROBE_FEAT_DIR = (
    FINAL_DIR
    / "probe_features"
    / "method2"
    / DIM_TAG
)

TARGET_DIR = (
    FINAL_DIR
    / "targets"
    / "method2"
    / DIM_TAG
)

PROBE_FEAT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

def output_paths(lp_probe_nodes):
    lp_probe_nodes = int(lp_probe_nodes)

    probe_feat_file = (
        PROBE_FEAT_DIR
        / (
            f"probe_features_method2_{DIM_TAG}"
            f"_probe{lp_probe_nodes}.csv"
        )
    )

    target_file = (
        TARGET_DIR
        / (
            f"targets_method2_relaxation_{DIM_TAG}"
            f"_probe{lp_probe_nodes}.csv"
        )
    )

    return probe_feat_file, target_file


KEY_COL = "instance_key"


# ============================================================
# HELPERS
# ============================================================

def _ts():
    return datetime.now().strftime("%H:%M:%S")


def read_any(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"File not found: {path}"
        )

    suffix = path.suffix.lower()

    if suffix == ".parquet":
        return pd.read_parquet(path)

    if suffix in {
        ".xlsx",
        ".xls",
    }:
        return pd.read_excel(path)

    if suffix == ".csv":
        return pd.read_csv(path)

    raise ValueError(
        f"Unsupported: {suffix}"
    )


def _require_columns(
    df,
    required_cols,
    name,
):
    missing = [
        c
        for c in required_cols
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{name} missing columns: {missing}\n"
            f"Available: {list(df.columns)}"
        )


def _get_status_col(df):
    if "status" in df.columns:
        return "status"

    if "node_status" in df.columns:
        return "node_status"

    raise ValueError(
        "No status column found."
    )


def _norm_status(value):
    if value is None:
        return ""

    if (
        isinstance(
            value,
            float,
        )
        and np.isnan(
            value
        )
    ):
        return ""

    return str(
        value
    ).strip().lower()


def _text_column(
    df,
    column,
):
    if column not in df.columns:
        return pd.Series(
            "",
            index=df.index,
            dtype="object",
        )

    return (
        df[column]
        .fillna("")
        .astype(str)
        .str.lower()
    )


def _safe_div(
    num,
    den,
):
    if (
        np.isfinite(num)
        and np.isfinite(den)
        and abs(den) > EPS
    ):
        return (
            float(num)
            / float(den)
        )

    return np.nan


def _safe_signed_div(
    num,
    den,
    eps,
):
    if (
        not np.isfinite(num)
        or not np.isfinite(den)
    ):
        return np.nan

    den_adj = (
        den
        + np.sign(den)
        * eps
        if den != 0.0
        else eps
    )

    return float(
        num
        / den_adj
    )


def _norm_by_root(
    value,
    root_obj,
):
    if (
        not np.isfinite(value)
        or not np.isfinite(root_obj)
    ):
        return np.nan

    return (
        float(
            value
            - root_obj
        )
        / float(
            abs(root_obj)
            + EPS
        )
    )


def _status_from_bound(
    value,
):
    return (
        "optimal"
        if np.isfinite(value)
        else "unknown"
    )


def _canon_name(x):
    s = str(x).strip()

    if s.lower().endswith(".pkl"):
        s = s[:-4]

    return s


def _make_key(
    source,
    instance_name,
):
    return (
        f"{str(source).upper()}||"
        f"{_canon_name(instance_name)}"
    )


# ============================================================
# CLEAN-LIVE VALIDATION
# ============================================================

def _find_unresolved_instance_keys(
    df_live,
):
    status_text = _text_column(
        df_live,
        "status",
    )

    solve_status_text = _text_column(
        df_live,
        "solve_status",
    )

    note_text = _text_column(
        df_live,
        "note",
    )

    unresolved_mask = (
        status_text.isin(
            {
                "unknown",
                "error",
                "unbounded",
            }
        )
        | solve_status_text.str.contains(
            "unknown",
            na=False,
        )
        | solve_status_text.str.contains(
            "unbounded",
            na=False,
        )
        | solve_status_text.str.contains(
            "error",
            na=False,
        )
        | note_text.str.contains(
            "unresolved solver status",
            na=False,
        )
        | note_text.str.contains(
            "unexpected unbounded",
            na=False,
        )
        | (
            solve_status_text.str.startswith(
                "gurobi_status_"
            )
            & ~solve_status_text.eq(
                "gurobi_status_4"
            )
        )
    )

    return sorted(
        df_live.loc[
            unresolved_mask,
            KEY_COL,
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )


def _require_clean_live_file(
    df_live,
):
    _require_columns(
        df_live,
        [
            KEY_COL,
            "source",
            "instance_index",
            "instance_name",
            "relaxation",
            "obj",
            "depth",
            "status",
            "processed_count",
            "global_frontier_lb_after_step",
            "n",
        ],
        "Combined LIVE file",
    )

    unresolved = (
        _find_unresolved_instance_keys(
            df_live
        )
    )

    if unresolved:
        raise ValueError(
            "LIVE file contains unresolved instances. "
            "Run the combined post-run filter first. "
            f"First keys: {unresolved[:30]}"
        )

    n_num = pd.to_numeric(
        df_live["n"],
        errors="coerce",
    )

    if n_num.isna().any():
        raise ValueError(
            "Combined LIVE has missing/non-numeric n."
        )

    if MAX_N_ALLOWED is not None and (n_num > MAX_N_ALLOWED).any():
        bad = (
            df_live.loc[
                n_num
                > MAX_N_ALLOWED,
                [
                    KEY_COL,
                    "n",
                ],
            ]
            .drop_duplicates()
        )

        raise RuntimeError(
            f"Combined LIVE contains n > {MAX_N_ALLOWED}:\n"
            + bad.head(30).to_string(
                index=False
            )
        )


# ============================================================
# FRONTIER-LB
# ============================================================

def compute_true_leaf_bound(
    df_relax,
    budget,
):
    _require_columns(
        df_relax,
        [
            "processed_count",
            "global_frontier_lb_after_step",
        ],
        "LIVE relaxation tree",
    )

    eligible = (
        df_relax[
            (
                df_relax[
                    "processed_count"
                ]
                <= int(budget)
            )
            & df_relax[
                "global_frontier_lb_after_step"
            ].notna()
        ]
        .sort_values(
            "processed_count"
        )
    )

    if eligible.empty:
        return np.nan

    return float(
        eligible.iloc[
            -1
        ][
            "global_frontier_lb_after_step"
        ]
    )


# ============================================================
# STATISTICS
# ============================================================

def compute_statistics(
    vector,
    stat_names=None,
):
    if stat_names is None:
        stat_names = STAT_NAMES

    if (
        vector is None
        or len(vector) == 0
    ):
        return {
            n: np.nan
            for n in stat_names
        }

    values = np.asarray(
        vector,
        dtype=float,
    ).reshape(-1)

    values = values[
        np.isfinite(
            values
        )
    ]

    if values.size == 0:
        return {
            n: np.nan
            for n in stat_names
        }

    mean_val = float(
        np.mean(values)
    )

    std_val = float(
        np.std(values)
    )

    mu3 = (
        float(
            np.mean(
                (
                    values
                    - mean_val
                )
                ** 3
            )
        )
        if values.size > 1
        else 0.0
    )

    q1, q3 = np.percentile(
        values,
        [
            25,
            75,
        ],
    )

    iqr = float(
        q3
        - q1
    )

    pout = (
        float(
            np.mean(
                (
                    values
                    < q1
                    - 1.5
                    * iqr
                )
                | (
                    values
                    > q3
                    + 1.5
                    * iqr
                )
            )
        )
        if iqr > 0
        else 0.0
    )

    all_stats = {
        "min": float(
            np.min(values)
        ),
        "max": float(
            np.max(values)
        ),
        "avg": mean_val,
        "mu3": mu3,
        "IQR": iqr,
        "Pout": pout,
        "CV": _safe_signed_div(
            std_val,
            mean_val,
            EPS_CV,
        ),
    }

    return {
        n: all_stats[n]
        for n in stat_names
    }


def _add_stats_features(
    feats,
    prefix,
    vector_name,
    vector,
    stat_names=None,
):
    if stat_names is None:
        stat_names = STAT_NAMES

    stats = compute_statistics(
        vector,
        stat_names=stat_names,
    )

    for s in stat_names:
        feats[
            f"h_{s}_{prefix}_{vector_name}"
        ] = stats[s]


# ============================================================
# PROBE FEATURES
# ============================================================

def extract_probe_features(
    df_relax,
    n_nodes,
    root_obj,
    prefix,
):
    _require_columns(
        df_relax,
        [
            "processed_count",
            "global_frontier_lb_after_step",
        ],
        "LIVE relaxation tree",
    )

    status_col = (
        _get_status_col(
            df_relax
        )
    )

    df_window = (
        df_relax
        .sort_values(
            "processed_count"
        )
        .iloc[
            : int(
                max(
                    1,
                    n_nodes,
                )
            )
        ]
        .copy()
    )

    n_total = len(
        df_window
    )

    feats = {}

    if n_total == 0:
        for k in [
            f"{prefix}_infeas_frac_probe",
            f"{prefix}_max_depth_probe",
            f"{prefix}_improvement_rate_rel",
            f"{prefix}_bound_pruned_frac_probe",
            f"{prefix}_frontier_improvement_rel",
        ]:
            feats[k] = np.nan

        _add_stats_features(
            feats,
            prefix,
            "obj_improvement_rel",
            [],
            stat_names=STAT_NAMES,
        )

        _add_stats_features(
            feats,
            prefix,
            "frontier_improvement_rel",
            [],
            stat_names=FRONTIER_STAT_NAMES,
        )

        return (
            feats,
            np.nan,
        )

    status_norm = (
        df_window[
            status_col
        ]
        .map(
            _norm_status
        )
    )

    infeas_mask = (
        status_norm.eq(
            "infeasible"
        )
    )

    bound_pruned_mask = (
        status_norm.eq(
            "pruned_by_bound"
        )
    )

    valid_obj_rows = (
        df_window[
            status_norm.isin(
                [
                    "processed",
                    "pruned",
                ]
            )
        ]
        .copy()
    )

    objs = (
        pd.to_numeric(
            valid_obj_rows[
                "obj"
            ],
            errors="coerce",
        )
        .dropna()
        .astype(float)
        .values
    )

    depths = (
        pd.to_numeric(
            df_window[
                "depth"
            ],
            errors="coerce",
        )
        .dropna()
        .astype(float)
        .values
    )

    feats[
        f"{prefix}_infeas_frac_probe"
    ] = float(
        infeas_mask.sum()
        / n_total
    )

    feats[
        f"{prefix}_max_depth_probe"
    ] = (
        int(
            np.max(
                depths
            )
        )
        if len(depths) > 0
        else np.nan
    )

    obj_impr = [
        x
        for x in [
            _norm_by_root(
                float(o),
                root_obj,
            )
            for o in objs
            if np.isfinite(o)
        ]
        if np.isfinite(x)
    ]

    _add_stats_features(
        feats,
        prefix,
        "obj_improvement_rel",
        obj_impr,
        stat_names=STAT_NAMES,
    )

    feats[
        f"{prefix}_improvement_rate_rel"
    ] = (
        _safe_div(
            float(
                np.max(objs)
            )
            - root_obj,
            (
                abs(root_obj)
                + EPS
            )
            * max(
                len(
                    valid_obj_rows
                ),
                1,
            ),
        )
        if (
            len(objs) > 0
            and np.isfinite(
                root_obj
            )
        )
        else np.nan
    )

    feats[
        f"{prefix}_bound_pruned_frac_probe"
    ] = float(
        bound_pruned_mask.sum()
        / n_total
    )

    frontier_values = (
        pd.to_numeric(
            df_window[
                "global_frontier_lb_after_step"
            ],
            errors="coerce",
        )
        .dropna()
        .astype(float)
        .values
    )

    frontier_impr = [
        x
        for x in [
            _norm_by_root(
                float(v),
                root_obj,
            )
            for v in frontier_values
            if np.isfinite(v)
        ]
        if np.isfinite(x)
    ]

    _add_stats_features(
        feats,
        prefix,
        "frontier_improvement_rel",
        frontier_impr,
        stat_names=FRONTIER_STAT_NAMES,
    )

    if len(
        frontier_values
    ) > 0:
        probe_frontier_lb = float(
            frontier_values[
                -1
            ]
        )

        feats[
            f"{prefix}_frontier_improvement_rel"
        ] = _norm_by_root(
            probe_frontier_lb,
            root_obj,
        )

    else:
        probe_frontier_lb = (
            np.nan
        )

        feats[
            f"{prefix}_frontier_improvement_rel"
        ] = np.nan

    return (
        feats,
        probe_frontier_lb,
    )


# ============================================================
# METHOD 1 BUDGET FILE
# ============================================================

def prepare_method1_budget_df(
    df_budget,
    lp_probe_nodes,
):
    df_budget = (
        df_budget.copy()
    )

    _require_columns(
        df_budget,
        [
            KEY_COL,
            "lp_time_pred",
            "sdp_time_pred",
            "lp_nodes",
            "sdp_nodes",
        ],
        "Method 1 budget file",
    )

    df_budget[
        KEY_COL
    ] = (
        df_budget[
            KEY_COL
        ]
        .astype(str)
        .str.strip()
    )

    for col in [
        "lp_time_pred",
        "sdp_time_pred",
        "lp_nodes",
        "sdp_nodes",
    ]:
        df_budget[
            col
        ] = pd.to_numeric(
            df_budget[
                col
            ],
            errors="coerce",
        )

    if df_budget[
        [
            KEY_COL,
            "lp_time_pred",
            "sdp_time_pred",
            "lp_nodes",
            "sdp_nodes",
        ]
    ].isna().any().any():
        raise ValueError(
            "Method 1 budget file has missing values."
        )

    if df_budget[
        KEY_COL
    ].duplicated().any():
        raise ValueError(
            "Method 1 budget file has "
            "duplicate instance_key values."
        )

    if (
        df_budget[
            [
                "lp_nodes",
                "sdp_nodes",
            ]
        ]
        <= 0
    ).any().any():
        raise ValueError(
            "Method 1 budget file has "
            "non-positive node budgets."
        )

    ratio = (
        df_budget[
            "sdp_nodes"
        ]
        / df_budget[
            "lp_nodes"
        ].clip(
            lower=1
        )
    )

    df_budget[
        "sdp_probe_nodes"
    ] = (
        np.floor(
            ratio
            * lp_probe_nodes
        )
        .clip(
            1,
            lp_probe_nodes,
        )
        .astype(int)
    )

    df_budget[
        "predicted_time_ratio_lp_over_sdp"
    ] = (
        df_budget[
            "lp_time_pred"
        ].clip(
            lower=1e-6
        )
        / df_budget[
            "sdp_time_pred"
        ].clip(
            lower=1e-6
        )
    )

    df_budget[
        "predicted_time_ratio_sdp_over_lp"
    ] = (
        df_budget[
            "sdp_time_pred"
        ].clip(
            lower=1e-6
        )
        / df_budget[
            "lp_time_pred"
        ].clip(
            lower=1e-6
        )
    )

    df_budget[
        "sdp_node_fraction"
    ] = (
        df_budget[
            "sdp_nodes"
        ]
        / df_budget[
            "lp_nodes"
        ].clip(
            lower=1
        )
    )

    return (
        df_budget
        .sort_values(
            KEY_COL
        )
        .reset_index(drop=True)
    )


# ============================================================
# STATIC 64 FEATURES
# ============================================================

def prepare_static_features(
    df_features,
):
    df_features = (
        df_features.copy()
    )

    _require_columns(
        df_features,
        [
            "id",
            "instance_name",
        ],
        "Static feature file",
    )

    # Synthetic′ features are keyed by numeric original instance IDs.
    # Match post-run LIVE key: SYNTHETIC||<numeric id>.
    ids = pd.to_numeric(df_features["id"], errors="raise")
    if ids.isna().any() or not (ids % 1 == 0).all():
        raise ValueError("Synthetic static feature IDs must be integers")
    df_features["id"] = ids.astype("int64")
    df_features["source"] = "SYNTHETIC"
    df_features[KEY_COL] = "SYNTHETIC||" + df_features["id"].astype(str)
    if df_features[KEY_COL].duplicated().any():
        raise ValueError("Duplicate Synthetic′ instance IDs in static features")

    # Exact static schema audit: id + instance_name + 64 ML features.
    non_ml = {
        "id",
        "source",
        "instance_name",
        KEY_COL,
    }

    ml_cols = [
        c
        for c in df_features.columns
        if c not in non_ml
    ]

    if len(
        ml_cols
    ) != 64:
        raise ValueError(
            f"Expected exactly 64 static ML features, "
            f"found {len(ml_cols)}."
        )

    return df_features


def clean_static_feature_row(
    row_dict,
):
    remove_cols = {
        "id",
        "source",
        KEY_COL,
        "instance_id",
        "instance_index",
        "instance_name",
        "name",
        "problem_name",
    }

    return {
        k: v
        for k, v in row_dict.items()
        if k not in remove_cols
    }


# ============================================================
# TARGET
# ============================================================

def make_target_from_bounds(
    lp_obj,
    sdp_obj,
):
    if (
        not np.isfinite(
            lp_obj
        )
        or not np.isfinite(
            sdp_obj
        )
    ):
        return (
            np.nan,
            "unknown",
            np.nan,
        )

    delta = _safe_div(
        lp_obj
        - sdp_obj,
        abs(lp_obj)
        + abs(sdp_obj)
        + ETA,
    )

    if not np.isfinite(
        delta
    ):
        return (
            np.nan,
            "unknown",
            np.nan,
        )

    if delta > -TAU_BINARY:
        return (
            1,
            "LP",
            delta,
        )

    return (
        0,
        "SDP_PRIME",
        delta,
    )


# ============================================================
# MAIN EXTRACTION
# ============================================================

def extract_method2_data(
    df_live,
    df_features,
    df_budget,
    lp_probe_nodes,
):
    _require_clean_live_file(
        df_live
    )

    df_live = (
        df_live.copy()
    )

    df_live[
        KEY_COL
    ] = (
        df_live[
            KEY_COL
        ]
        .astype(str)
        .str.strip()
    )

    df_features = (
        prepare_static_features(
            df_features
        )
    )

    df_budget = (
        prepare_method1_budget_df(
            df_budget,
            lp_probe_nodes,
        )
    )

    # --------------------------------------------------------
    # EXACT SET ALIGNMENT
    # --------------------------------------------------------
    live_keys = set(
        df_live[
            KEY_COL
        ]
        .unique()
        .tolist()
    )

    feature_keys = set(
        df_features[
            KEY_COL
        ]
        .unique()
        .tolist()
    )

    budget_keys = set(
        df_budget[
            KEY_COL
        ]
        .unique()
        .tolist()
    )

    if live_keys != feature_keys:
        raise RuntimeError(
            "Combined LIVE and 64-feature instance sets differ. "
            f"Missing features={sorted(live_keys-feature_keys)[:20]}, "
            f"extra features={sorted(feature_keys-live_keys)[:20]}"
        )

    if live_keys != budget_keys:
        raise RuntimeError(
            "Combined LIVE and Method 1 budget instance sets differ. "
            f"Missing budgets={sorted(live_keys-budget_keys)[:20]}, "
            f"extra budgets={sorted(budget_keys-live_keys)[:20]}"
        )

    budget_lookup = {
        str(
            r[
                KEY_COL
            ]
        ): r
        for _,
        r in df_budget.iterrows()
    }

    feature_lookup = {
        str(
            r[
                KEY_COL
            ]
        ): r
        for _,
        r in df_features.iterrows()
    }

    print(
        f"[{_ts()}] LP probe nodes : "
        f"{lp_probe_nodes}",
        flush=True,
    )

    probe_rows = []
    target_rows = []

    for (
        instance_key,
        df_inst,
    ) in df_live.groupby(
        KEY_COL,
        dropna=True,
        sort=True,
    ):
        instance_key = str(
            instance_key
        )

        if (
            instance_key
            not in budget_lookup
        ):
            raise ValueError(
                f"Budget missing for "
                f"{instance_key}."
            )

        if (
            instance_key
            not in feature_lookup
        ):
            raise ValueError(
                f"Static features missing for "
                f"{instance_key}."
            )

        source = str(
            df_inst[
                "source"
            ].iloc[0]
        ).upper()

        instance_name = str(
            df_inst[
                "instance_name"
            ].iloc[0]
        )

        instance_index = int(
            pd.to_numeric(
                df_inst[
                    "instance_index"
                ].iloc[0],
                errors="raise",
            )
        )

        budget_row = (
            budget_lookup[
                instance_key
            ]
        )

        lp_nodes = int(
            budget_row[
                "lp_nodes"
            ]
        )

        sdp_nodes = int(
            budget_row[
                "sdp_nodes"
            ]
        )

        sdp_probe_nodes = int(
            budget_row[
                "sdp_probe_nodes"
            ]
        )

        df_lp = (
            df_inst[
                df_inst[
                    "relaxation"
                ]
                == "LP"
            ]
            .copy()
        )

        df_sdp = (
            df_inst[
                df_inst[
                    "relaxation"
                ]
                == "SDP_PRIME"
            ]
            .copy()
        )

        if (
            len(df_lp) == 0
            or len(df_sdp) == 0
        ):
            raise ValueError(
                f"Missing LP or SDP_PRIME tree "
                f"for {instance_key}."
            )

        lp_root_rows = (
            df_lp[
                df_lp[
                    "depth"
                ]
                == 0
            ]
            .sort_values(
                "processed_count"
            )
        )

        sdp_root_rows = (
            df_sdp[
                df_sdp[
                    "depth"
                ]
                == 0
            ]
            .sort_values(
                "processed_count"
            )
        )

        if (
            len(lp_root_rows) != 1
            or len(sdp_root_rows) != 1
        ):
            raise ValueError(
                "Expected exactly one root row "
                f"per relaxation for {instance_key}."
            )

        lp_root_obj = pd.to_numeric(
            lp_root_rows[
                "obj"
            ].iloc[0],
            errors="coerce",
        )

        sdp_root_obj = pd.to_numeric(
            sdp_root_rows[
                "obj"
            ].iloc[0],
            errors="coerce",
        )

        if (
            not np.isfinite(
                lp_root_obj
            )
            or not np.isfinite(
                sdp_root_obj
            )
        ):
            raise ValueError(
                f"Non-finite root objective "
                f"for {instance_key}."
            )

        (
            lp_probe_feats,
            lp_probe_frontier_lb,
        ) = extract_probe_features(
            df_lp,
            lp_probe_nodes,
            float(
                lp_root_obj
            ),
            "lp",
        )

        (
            sdp_probe_feats,
            sdp_probe_frontier_lb,
        ) = extract_probe_features(
            df_sdp,
            sdp_probe_nodes,
            float(
                sdp_root_obj
            ),
            "sdp",
        )

        if (
            np.isfinite(
                lp_probe_frontier_lb
            )
            and np.isfinite(
                sdp_probe_frontier_lb
            )
        ):
            relative_frontier_gap_probe = (
                _safe_div(
                    lp_probe_frontier_lb
                    - sdp_probe_frontier_lb,
                    abs(
                        lp_probe_frontier_lb
                    )
                    + abs(
                        sdp_probe_frontier_lb
                    )
                    + ETA,
                )
            )

            probe_lp_better = int(
                lp_probe_frontier_lb
                >= sdp_probe_frontier_lb
            )

        else:
            relative_frontier_gap_probe = (
                np.nan
            )

            probe_lp_better = (
                np.nan
            )

        spectral_feats = (
            clean_static_feature_row(
                feature_lookup[
                    instance_key
                ].to_dict()
            )
        )

        lp_leaf_bound = (
            compute_true_leaf_bound(
                df_lp,
                budget=lp_nodes,
            )
        )

        sdp_leaf_bound = (
            compute_true_leaf_bound(
                df_sdp,
                budget=sdp_nodes,
            )
        )

        (
            y,
            winner,
            delta,
        ) = make_target_from_bounds(
            lp_leaf_bound,
            sdp_leaf_bound,
        )

        if not np.isfinite(
            y
        ):
            raise ValueError(
                "Could not create finite target "
                f"for {instance_key}."
            )

        common_meta = {
            KEY_COL:
                instance_key,
            "source":
                source,
            "instance_index":
                instance_index,
            "instance_name":
                instance_name,
        }

        probe_rows.append(
            {
                **common_meta,
                **lp_probe_feats,
                **sdp_probe_feats,
                "relative_frontier_gap_probe":
                    relative_frontier_gap_probe,
                "probe_lp_better":
                    probe_lp_better,
                **spectral_feats,
            }
        )

        target_rows.append(
            {
                **common_meta,
                # Collision-safe target ID.
                "id":
                    instance_key,
                "y":
                    y,
                "winner":
                    winner,
                "delta":
                    delta,
                "lp_obj":
                    lp_leaf_bound,
                "sdp_prime_obj":
                    sdp_leaf_bound,
                "lp_status":
                    _status_from_bound(
                        lp_leaf_bound
                    ),
                "sdp_prime_status":
                    _status_from_bound(
                        sdp_leaf_bound
                    ),
                "lp_nodes":
                    lp_nodes,
                "sdp_nodes":
                    sdp_nodes,
                "lp_probe_nodes":
                    lp_probe_nodes,
                "sdp_probe_nodes":
                    sdp_probe_nodes,
                "lp_time_pred":
                    float(
                        budget_row[
                            "lp_time_pred"
                        ]
                    ),
                "sdp_time_pred":
                    float(
                        budget_row[
                            "sdp_time_pred"
                        ]
                    ),
                "predicted_time_ratio_lp_over_sdp":
                    float(
                        budget_row[
                            "predicted_time_ratio_lp_over_sdp"
                        ]
                    ),
                "predicted_time_ratio_sdp_over_lp":
                    float(
                        budget_row[
                            "predicted_time_ratio_sdp_over_lp"
                        ]
                    ),
                "sdp_node_fraction":
                    float(
                        budget_row[
                            "sdp_node_fraction"
                        ]
                    ),
            }
        )

    if (
        not probe_rows
        or not target_rows
    ):
        raise ValueError(
            "Method 2 extraction "
            "produced empty output."
        )

    df_probe = (
        pd.DataFrame(
            probe_rows
        )
        .sort_values(
            KEY_COL
        )
        .reset_index(drop=True)
    )

    df_target = (
        pd.DataFrame(
            target_rows
        )
        .sort_values(
            KEY_COL
        )
        .reset_index(drop=True)
    )

    if len(
        df_probe
    ) != len(
        df_target
    ):
        raise RuntimeError(
            "Feature and target row counts differ."
        )

    if not df_probe[
        KEY_COL
    ].equals(
        df_target[
            KEY_COL
        ]
    ):
        raise RuntimeError(
            "Feature and target instance_key "
            "sequences do not match."
        )

    return (
        df_probe,
        df_target,
    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    PROBE_FEAT_DIR.mkdir(parents=True, exist_ok=True)
    TARGET_DIR.mkdir(parents=True, exist_ok=True)
    print("=" * 72)
    print(
        f"[{_ts()}] METHOD 2 — "
        "EXTRACT COMBINED TRAINING DATA"
    )
    print(
        f"[{_ts()}] DIM_TAG          : "
        f"{DIM_TAG}"
    )
    print(
        f"[{_ts()}] LIVE             : "
        f"{LIVE_PATH}"
    )
    print(
        f"[{_ts()}] Static 64 features: "
        f"{FEATURES_PATH}"
    )
    print(
        f"[{_ts()}] Method 1 budgets : "
        f"{METHOD1_BUDGET_PATH}"
    )
    print(
        f"[{_ts()}] Probe windows    : "
        f"{PROBE_WINDOWS}"
    )
    print("=" * 72)

    # Validate once before loading the data.
    probe_windows = []
    for value in PROBE_WINDOWS:
        try:
            value = int(value)
        except Exception as exc:
            raise ValueError(
                f"Invalid probe-window value: {value!r}"
            ) from exc

        if value <= 0:
            raise ValueError(
                f"Probe windows must be positive integers. Found: {value}"
            )

        if value not in probe_windows:
            probe_windows.append(value)

    if not probe_windows:
        raise ValueError(
            "PROBE_WINDOWS is empty. Add at least one value, e.g. [10, 20]."
        )

    df_live = read_any(
        LIVE_PATH
    )

    df_features = read_any(
        FEATURES_PATH
    )

    df_budget = read_any(
        METHOD1_BUDGET_PATH
    )

    print(
        f"[{_ts()}] "
        f"LIVE rows={len(df_live):,}  "
        f"features={len(df_features):,}  "
        f"budget={len(df_budget):,}",
        flush=True,
    )

    created_files = []

    for lp_probe_nodes in probe_windows:
        print("\n" + "=" * 72)
        print(
            f"[{_ts()}] START PROBE WINDOW = "
            f"{lp_probe_nodes}"
        )
        print("=" * 72)

        (
            df_probe,
            df_target,
        ) = extract_method2_data(
            df_live,
            df_features,
            df_budget,
            lp_probe_nodes=lp_probe_nodes,
        )

        (
            probe_feat_file,
            target_file,
        ) = output_paths(
            lp_probe_nodes
        )

        df_probe.to_csv(
            probe_feat_file,
            index=False,
        )

        df_target.to_csv(
            target_file,
            index=False,
        )

        created_files.append(
            (
                lp_probe_nodes,
                probe_feat_file,
                target_file,
                len(df_probe),
                len(df_probe.columns),
                len(df_target),
                len(df_target.columns),
            )
        )

        print(
            f"\n[{_ts()}] Probe features -> "
            f"{probe_feat_file}  "
            f"rows={len(df_probe):,}  "
            f"cols={len(df_probe.columns):,}"
        )

        print(
            f"[{_ts()}] Targets -> "
            f"{target_file}  "
            f"rows={len(df_target):,}  "
            f"cols={len(df_target.columns):,}"
        )

        print(
            f"\n[{_ts()}] Rows by source:"
        )
        print(
            df_target[
                "source"
            ].value_counts().to_string()
        )

        print(
            f"\n[{_ts()}] Label distribution:"
        )
        print(
            df_target[
                "winner"
            ].value_counts(
                dropna=False
            ).to_string()
        )

        print(
            f"\n[{_ts()}] Missing labels: "
            f"{int(df_target['y'].isna().sum())}"
        )

    print("\n" + "=" * 72)
    print(
        f"[{_ts()}] ALL REQUESTED PROBE WINDOWS COMPLETED"
    )

    for (
        lp_probe_nodes,
        probe_feat_file,
        target_file,
        n_probe_rows,
        n_probe_cols,
        n_target_rows,
        n_target_cols,
    ) in created_files:
        print(
            f"[{_ts()}] probe={lp_probe_nodes}: "
            f"features={probe_feat_file.name} "
            f"({n_probe_rows}x{n_probe_cols}) | "
            f"targets={target_file.name} "
            f"({n_target_rows}x{n_target_cols})"
        )

    print("=" * 72)


[21:35:50] METHOD 2 — EXTRACT COMBINED TRAINING DATA
[21:35:50] DIM_TAG          : combined_minlplib_qplib
[21:35:50] LIVE             : C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\combined_minlplib_qplib_BB_LIVE_clean.parquet
[21:35:50] Static 64 features: C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\combined_minlplib_qplib_features_DI_64_clean.xlsx
[21:35:50] Method 1 budgets : C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\reg_outputs_method1\combined_minlplib_qplib\DI__method1_time_regression\method1_budget_predictions_combined_minlplib_qplib.csv
[21:35:50] Probe windows    : [10, 20, 30]
[21:35:52] LIVE rows=45,045  features=283  budget=283

[21:35:52] START PROBE WINDOW = 10
[21:35:52] LP probe nodes : 10

[21:35:53] Probe features -> C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\probe_features\method2\combined_minlplib_qplib\probe_features_method2_combined_minlplib_qplib_probe10.csv  rows=283  cols=106